# YouTube Video Summarizer — Gemini (Ed Donner-style API)

An original beginner project: enter a public YouTube URL, retrieve available captions, and summarize important concepts using Gemini through the OpenAI-compatible Chat Completions API.

**Note:** This summarizes spoken captions, not video visuals. Videos without accessible transcripts may not work.

## 1. Install libraries
Run this cell once in your notebook. `%pip` works inside a Jupyter code cell, not PowerShell.

In [1]:
%pip install -U openai python-dotenv youtube-transcript-api

Note: you may need to restart the kernel to use updated packages.


## 2. Load your Gemini API key
Create a `.env` file in your project folder containing `GOOGLE_API_KEY=your_key`. Keep `.env` out of Git using `.gitignore`. Restart/run this cell after editing `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY not found. Add it to your .env file.")
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
print("Gemini client ready!")

Gemini client ready!


## 3. User enters a YouTube URL
When you run this cell, paste the link in the input box and press Enter.

In [3]:
youtube_url = input("Paste a public YouTube video URL: ").strip()
print("You entered:", youtube_url)

You entered: https://www.youtube.com/watch?v=lgBLqo8TwgE&t=289s


## 4. Extract the video ID and transcript
The URL identifies the video. The transcript API retrieves captions when YouTube makes them accessible. `video_text` is created here, before the Gemini API call.

In [4]:
from urllib.parse import urlparse, parse_qs
from youtube_transcript_api import YouTubeTranscriptApi

parsed_url = urlparse(youtube_url)
host = (parsed_url.hostname or "").lower()

if host in ("youtu.be", "www.youtu.be"):
    video_id = parsed_url.path.strip("/").split("/")[0]
elif host in ("youtube.com", "www.youtube.com", "m.youtube.com"):
    if parsed_url.path == "/watch":
        video_id = parse_qs(parsed_url.query).get("v", [None])[0]
    elif parsed_url.path.startswith(("/shorts/", "/live/")):
        video_id = parsed_url.path.strip("/").split("/")[1]
    else:
        video_id = None
else:
    video_id = None

if not video_id:
    raise ValueError("Please enter a supported YouTube video URL.")

transcript = YouTubeTranscriptApi().fetch(video_id)
video_text = " ".join(snippet.text for snippet in transcript)
print("Transcript ready! Characters:", len(video_text))
print(video_text[:350])

Transcript ready! Characters: 107175
A man [music] who had a wife he was crazy about, but he was wildly jealous. And one day they were coming into the apartment building and there was a building manager. And she looked at him and said, "Oh, hello." And they went into the house and he went insane. He started screaming. He thought she was flirting. And so he stabbed her [screaming] so m


## 5. Write the prompts
The system prompt defines the summarizer’s behavior; the user prompt gives the specific task.

In [5]:
system_prompt = """You are a helpful educational YouTube transcript summarizer.
Focus on concepts, important explanations, examples, and actionable takeaways.
Skip greetings, sponsor messages, filler, and repetition.
Do not invent details not present in the transcript.
Organize the result with clear headings and simple language."""

user_prompt = "Summarize the important ideas. Explain each key concept clearly, include examples if discussed, and end with key takeaways."

## 6. Ask Gemini — Ed Donner-style Chat Completions
This is a **text** request containing the transcript. Do not send `video_url`: the Gemini OpenAI-compatible endpoint rejected that content type.

In [6]:
response = gemini.chat.completions.create(
            model="gemini-3.1-flash-lite",
            messages=[
                {"role": "system", "content": system_prompt},
                {
                    "role": "user",
                    "content": f"{user_prompt}\n\nYouTube transcript:\n{video_text}"
                }
            ]
        )

summary = response.choices[0].message.content or "No summary was returned."
print(summary)
        



This summary explores the professional life and insights of Barbara Butcher, a seasoned death investigator with over 5,500 cases of experience.

### The Role of a Death Investigator
Unlike a forensic pathologist who performs autopsies to find the *cause* of death (e.g., gunshot wound), a death investigator determines the **manner** of death (homicide, suicide, or accident).
*   **The Process:** The investigator examines the "totality of the scene"—the atmosphere, the room’s condition, and the body’s position—before physically touching the body.
*   **Evidence Collection:** They look for signs of struggle (flesh/hair under fingernails), analyze blood spatter, and assess the body for evidence of movement post-mortem.
*   **Collaboration:** Investigators work independently but closely with police to build a narrative that informs the pathologist, ensuring the autopsy is context-aware.

### Analyzing the Crime Scene
Barbara emphasizes that the body and the scene are the primary sources of 

## 7. What I learned
- Reading an API key securely from `.env`
- Taking input from a user
- Extracting a YouTube video ID and transcript
- Writing system and user prompts
- Calling Gemini with the OpenAI-compatible `chat.completions.create()` method
- Reading the answer with `response.choices[0].message.content`

**Limitations:** Some videos do not expose transcripts, and very long transcripts can exceed model input limits. The summary reflects caption text, not visual content.